# DQ STG GEO INTERVALS

Визуализация последнего прогона `dq-stg-geo-intervals` (логи `DQ_STG_GEO_INTERVALS`), профиль parquet `stg_geo_intervals` за отчётный день.

Перед запуском: `uv run mobile build-stg-geo-intervals`, затем `uv run mobile dq-stg-geo-intervals`.

In [ ]:
from __future__ import annotations

import pandas as pd
from IPython.display import display

from mobile.project_paths import PROJECT_ROOT

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 120)

ROOT = PROJECT_ROOT
TAG = "DQ_STG_GEO_INTERVALS"
BOUNDARY = "dataset_basic"

In [ ]:
from mobile.pipelines.nb.common import checks_by_status, load_dq_dashboard

try:
    dq_logs, latest, meta = load_dq_dashboard(ROOT, tag=TAG, boundary_check=BOUNDARY)
except (FileNotFoundError, ValueError) as exc:
    print(f"Нет DQ-логов для {TAG}: {exc}")
    latest = meta = None
else:
    print("Последний прогон:", meta)
    display(checks_by_status(latest))

## Метрики DQ (последний прогон)

Графики строятся только из полей `metrics` в логах `DQ_STG_GEO_INTERVALS`.

In [ ]:
import matplotlib.pyplot as plt

from mobile.pipelines.nb.common import (
    render_stg_geo_intervals_dq_cardinality,
    render_stg_geo_intervals_dq_gates,
    render_stg_geo_intervals_dq_nulls,
    render_stg_geo_intervals_dq_overview,
)

if latest is None:
    print("Нет данных для визуализации")
else:
    for renderer in (
        render_stg_geo_intervals_dq_overview,
        render_stg_geo_intervals_dq_gates,
        render_stg_geo_intervals_dq_nulls,
        render_stg_geo_intervals_dq_cardinality,
    ):
        fig = renderer(latest)
        plt.show()

In [ ]:
from mobile.pipelines.nb.common import failed_warning_table, metrics_wide_table

if latest is None:
    pass
else:
    display(pd.DataFrame([meta]))
    print("\n--- failed / warning ---")
    display(failed_warning_table(latest))
    print("\n--- все метрики (плоская таблица) ---")
    wide = metrics_wide_table(latest)
    display(wide.head(100))
    if len(wide) > 100:
        print(f"... всего строк metrics: {len(wide)}")

## Профиль parquet `stg_geo_intervals`

День берётся из последнего DQ-прогона (`dataset_basic.report_date`) или из самого свежего файла в `data/stg/geo_intervals/`.

In [ ]:
import matplotlib.pyplot as plt

from mobile.pipelines.nb.common import (
    _stg_geo_intervals_report_date,
    display_stg_geo_intervals_parquet_summary,
    render_stg_geo_intervals_parquet_profile,
)

report_date = _stg_geo_intervals_report_date(latest)
print(f"report_date: {report_date.isoformat()}")

try:
    display_stg_geo_intervals_parquet_summary(ROOT, report_date)
except FileNotFoundError as exc:
    print(exc)
else:
    fig = render_stg_geo_intervals_parquet_profile(ROOT, report_date)
    plt.show()